# 02 Generování oken z genomu (úkol 1.3)

Rozřeže genom na okna o délce `WINDOW`, vyřadí okna s více než 1 % N a rozdělí je
podle chromozomů: **trénink** chr1–22 a X kromě chr20 a chr21, **validace** chr20, **test** chr21.

Ukládají se jen souřadnice oken (`chrom`, `start`, `end`) a jejich GC obsah, podíl N
a podíl repetic. Sekvence se vyřezávají až při tokenizaci, takže stejná okna slouží
všem tokenizérům. Stačí CPU runtime, trvá to pár minut.

Předpoklad: kód z repa je pushnutý na GitHub a v Colab Secrets je `GITHUB_TOKEN` (úkol 0.6).

## 1. Nastavení

In [ ]:
WINDOW = 1024          # délka okna v bp
MAX_N_FRAC = 0.01      # max. podíl N v okně

DRIVE = "/content/drive/MyDrive/dna-lm"
FASTA_DRIVE = f"{DRIVE}/data/raw/genome/hg38.main.fa"
OUT_DIR = f"{DRIVE}/data/processed/windows_{WINDOW}"

REPO = "Valasik0/dna-lm"

## 2. Drive, kód z GitHubu

In [ ]:
import sys
assert "google.colab" in sys.modules or __import__("importlib").util.find_spec("google.colab"), \
    "Notebook neběží na Colab runtime: vpravo nahoře vyber kernel Colab"

from google.colab import drive
drive.mount("/content/drive")

# GitHub token: z Colab Secrets, a když to nejde (např. ve VS Code), ruční zadání
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print(f"Colab Secrets nedostupné ({type(e).__name__}), zadej token ručně:")
    from getpass import getpass
    token = getpass("GITHUB_TOKEN: ")

!git clone -q https://{token}@github.com/{REPO}.git /content/dna-lm 2>/dev/null || git -C /content/dna-lm pull -q
!pip install -q -e /content/dna-lm --no-deps
!pip install -q pyfaidx
!git -C /content/dna-lm log --oneline -1

## 3. Kopie genomu na lokální disk Colabu

Čtení 3 GB přímo z Drive je pomalé, lokální kopie se udělá jednou za sezení.

In [ ]:
import os, shutil
FASTA = "/content/genome/hg38.main.fa"
os.makedirs("/content/genome", exist_ok=True)
for ext in ["", ".fai"]:
    if not os.path.exists(FASTA + ext):
        shutil.copy(FASTA_DRIVE + ext, FASTA + ext)
!ls -lh /content/genome

## 4. Generování oken

In [ ]:
import time
from dnalm.data.genome import WindowConfig, make_windows

cfg = WindowConfig(window=WINDOW, max_n_frac=MAX_N_FRAC)
t0 = time.time()
summary = make_windows(FASTA, OUT_DIR, cfg)
print(f"Hotovo za {time.time() - t0:.0f} s")
!ls -lh {OUT_DIR}

## 5. Přehled

In [ ]:
import pandas as pd
from dnalm.data.genome import load_windows

rows = []
for split, info in summary["splits"].items():
    df = load_windows(f"{OUT_DIR}/{info['file']}")
    rows.append({
        "split": split,
        "chromozomy": ", ".join(info["chromosomes"]) if len(info["chromosomes"]) < 3 else f"{len(info['chromosomes'])} chromozomů",
        "posun_bp": info["stride"],
        "oken": len(df),
        "vyřazeno_N": info["windows_total"] - info["windows_kept"],
        "GC_průměr": round(df.gc_frac.mean(), 3),
        "repetice_průměr": round(df.repeat_frac.mean(), 3),
    })
overview = pd.DataFrame(rows).set_index("split")
overview.to_csv(f"{OUT_DIR}/overview.csv")
overview

## 6. Kontrola: sekvence několika oken

In [ ]:
from dnalm.data.genome import GenomeWindows

val = GenomeWindows(FASTA, f"{OUT_DIR}/val.tsv.gz")
print(len(val), "validačních oken")
for i in [0, len(val) // 2, len(val) - 1]:
    s = val[i]
    print(val.chrom[i], val.start[i], val.end[i], len(s), s[:60] + "...")
print("reverzní komplement:", val.get(0, reverse=True)[:60] + "...")